In [ ]:
from capsa.game import CapsaGame, display_cards
from capsa.players import HumanPlayer
from capsa.players.bots import RandomBot
from capsa.utils import generateCandidateMoves

players = [
    HumanPlayer("Player 1"),
    RandomBot("Player 2"),
    RandomBot("Player 3"),
    RandomBot("Player 4"),
]
game = CapsaGame(players)

game.state.lastPlayedCards = [
    (7, 0), (8, 1), (9, 2), (10, 3), (11, 0),
    # 8♦, 9♣, 10♥, J♠, Q♦
]
game.state.cards = [
    0, 4, 4, 4, 4, 4, 1, 0, 4, 4, 4, 4, 4,
    4, 0, 4, 4, 4, 4, 4, 4, 0, 4, 4, 0, 4,
    4, 4, 0, 4, 4, 4, 4, 4, 4, 0, 4, 4, 0,
    4, 4, 4, 0, 4, 4, 4, 4, 4, 4, 0, 4, 4,
]
game.player_turn = 0
game.state.playerPassFlag = [False, True, False, False]

In [ ]:
candidate_moves = generateCandidateMoves(0, game.state)

for i, move in enumerate(candidate_moves):
    print(f"{i}. ", end="")
    display_cards(move)

0. 
1. 8♦ 9♣ 10♥ J♠ Q♣ 
2. 9♣ 10♥ J♠ Q♣ K♥ 
3. 10♥ J♠ Q♣ K♥ A♦ 
4. J♠ Q♣ K♥ A♦ 2♣ 


### Model Train

In [ ]:
from capsa import CapsaGame, logging
from capsa.players.bots import StateEvalBot
logging.disable_log()

players = [
    StateEvalBot("Player 1", train=True),
    StateEvalBot("Player 2", train=True),
    StateEvalBot("Player 3", train=True),
    StateEvalBot("Player 4", train=True),
]
game = CapsaGame(players)

for i in range(1000):
    game.start()

    if i % 100 == 0:
        for player in players:
            print(player._loss, end=" | ")
        print()

In [2]:
for i, player in enumerate(players):
    player.save_model(f"./model_train/state_eval_s{i}.pt") 

### Group Trainer

In [1]:
from model_train import StateEvalGroupTrainer
from capsa import logging
logging.disable_log()

In [ ]:
trainer = StateEvalGroupTrainer()
trainer.train(1000)

In [3]:
trainer.save_model("./model_train/state_eval_group2.pt")

### Compute Average Rank

In [ ]:
from capsa import CapsaGame, logging
from capsa.players import HumanPlayer
from capsa.players.bots import RandomBot, StateEvalBot
logging.disable_log()

# players = [ StateEvalBot("Player 1"), StateEvalBot("Player 2"), StateEvalBot("Player 3"), StateEvalBot("Player 4") ] 
# players[0].load_model("./model_train/state_eval_s0.pt")
# players[1].load_model("./model_train/state_eval_s1.pt")
# players[2].load_model("./model_train/state_eval_s2.pt")
# players[3].load_model("./model_train/state_eval_s3.pt")

players = [ StateEvalBot("Player 1"), StateEvalBot("Player 2"), StateEvalBot("Player 3"), RandomBot("Player 4") ] 
players[0].load_model("./model_train/state_eval_group.pt")
players[1].load_model("./model_train/state_eval_s0.pt")
players[2].load_model("./model_train/state_eval_s3.pt")

In [5]:
ranks = [0, 0, 0, 0]
num_games = 1000

game = CapsaGame(players)

for _ in range(num_games):
    game.start()

    for i, r in enumerate(game.ranks):
        ranks[i] += r
        
ranks

[1303, 989, 1642, 2066]

### Test Play Model

In [ ]:
from capsa import CapsaGame, logging
from capsa.players import HumanPlayer
from capsa.players.bots import RandomBot, StateEvalBot
logging.enable_log()

players = [ 
    StateEvalBot("Player 1", show_cards=True),
    StateEvalBot("Player 2", show_cards=True),
    StateEvalBot("Player 3", show_cards=True),
    StateEvalBot("Player 4", show_cards=True),
] 

players[1].load_model("./model_train/state_eval_group.pt")
players[2].model = players[1].model
players[3].model = players[1].model
players[0].model = players[1].model

game = CapsaGame(players)
game.start()

[Player 1 Turn]

Cards in Hand:
3♦ K♦ 4♣ 5♣ 6♣ 7♣ K♣ 2♥ 5♥ 6♥ 7♠ 8♠ 9♠ 

Player 1 played: 3♦ 4♣ 5♥ 6♥ 7♣ 

[Player 2 Turn]

Cards in Hand:
5♦ 9♦ J♦ 2♣ 4♥ 7♥ 9♥ Q♥ K♥ A♠ 10♠ J♠ Q♠ 

Player 2 played: 4♥ 7♥ 9♥ Q♥ K♥ 

[Player 3 Turn]

Cards in Hand:
A♦ 4♦ 6♦ 7♦ 8♦ 9♣ 10♣ Q♣ 8♥ 10♥ J♥ 3♠ 5♠ 

Player 3 played: 4♦ 6♦ 7♦ 8♦ A♦ 

[Player 4 Turn]

Cards in Hand:
2♦ 10♦ Q♦ A♣ 3♣ 8♣ J♣ A♥ 3♥ 2♠ 4♠ 6♠ K♠ 

Player 4 played: 

[Player 1 Turn]

Cards in Hand:
K♦ 5♣ 6♣ K♣ 2♥ 7♠ 8♠ 9♠ 

Player 1 played: 

[Player 2 Turn]

Cards in Hand:
5♦ 9♦ J♦ 2♣ A♠ 10♠ J♠ Q♠ 

Player 2 played: 

[Player 3 Turn]

Cards in Hand:
9♣ 10♣ Q♣ 8♥ 10♥ J♥ 3♠ 5♠ 

Player 3 played: 8♥ 

[Player 4 Turn]

Cards in Hand:
2♦ 10♦ Q♦ A♣ 3♣ 8♣ J♣ A♥ 3♥ 2♠ 4♠ 6♠ K♠ 

Player 4 played: J♣ 

[Player 1 Turn]

Cards in Hand:
K♦ 5♣ 6♣ K♣ 2♥ 7♠ 8♠ 9♠ 

Player 1 played: K♦ 

[Player 2 Turn]

Cards in Hand:
5♦ 9♦ J♦ 2♣ A♠ 10♠ J♠ Q♠ 

Player 2 played: 

[Player 3 Turn]

Cards in Hand:
9♣ 10♣ Q♣ 10♥ J♥ 3♠ 5♠ 

Player 3 played: 

[Player 4 Turn]
